In [ ]:
# Install Transformer library
!pip install transformers torch

In [ ]:
#importing mmodel using transformer
from transformers import pipeline

In [ ]:
chatbot = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.1",
    pad_token_id=2,
    max_new_tokens=300
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
#prompt engineering to make the responses friendly and clear
SYSTEM_PROMPT = """
You are a helpful medical assistant chatbot. Act like a top 1% helpful medical assistants. Give simple and friendly health information.

Safety Rules:
- Do not diagnose diseases.
- Do not prescribe medicines.
- Do not give exact dosage or overdose limits.
- For medicine questions, do not simply say "yes". Say it may be used only as directed by a doctor/pharmacist or medicine label.
- For children, pregnancy, allergies, chronic illness, or serious symptoms, recommend consulting a doctor or pharmacist.
"""

danger_keywords = [
    "chest pain",
    "difficulty breathing",
    "suicide",
    "overdose",
    "severe bleeding"
]

In [ ]:
#create safety filter function
def safety_filter(query):
    query = query.lower()
    for word in danger_keywords:
        if word in query:
            return True
    return False

In [ ]:
def health_chatbot(user_query):

    if safety_filter(user_query):
        return "This may be serious. Please contact a doctor or emergency service immediately."

    prompt = f"""
    {SYSTEM_PROMPT}

    User: {user_query}

    Assistant:
    """

    response = chatbot(
        prompt,
        temperature=0.5,
        do_sample=True,
        return_full_text=False,
        eos_token_id=2
    )

    answer = response[0]["generated_text"].strip()

    # Remove extra generated chat
    stop_words = ["User:", "Assistant:"]

    for word in stop_words:
        if word in answer:
            answer = answer.split(word)[0]

    return answer.strip()

In [ ]:
# Create chatbot function
def health_chatbot(user_query):
  if safety_filter(user_query):
        return "This may be a serious medical emergency. Please contact a doctor or emergency service immediately."

  prompt = SYSTEM_PROMPT + "\nUser: " + user_query + "\nAssistant:"

  response = chatbot(
        prompt,
        do_sample=True,
        temperature=0.7,
        return_full_text=False
    )

  answer = response[0]["generated_text"].strip()

  return answer


In [ ]:
# Test Questions
questions = [
    "What causes a sore throat?",
    "Is paracetamol safe for children?"
]

for q in questions:

    print("\nUser:", q)
    print("Bot:", health_chatbot(q))
    print("-" * 50)

Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



User: What causes a sore throat?


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: Hello! I'm here to help you with any health-related questions you may have.

        A sore throat can be caused by a variety of factors. Some common causes include:

        - Viral infections: Cold viruses like rhinoviruses, coronaviruses, and adenoviruses are the most common causes of sore throats.
        - Bacterial infections: Strep throat is a bacterial infection that can cause a sore throat, and it's usually caused by the bacteria Streptococcus pyogenes.
        - Other infections: Other bacterial infections, such as tonsillitis, can also cause a sore throat.
        - Allergies: Allergies to dust, pollen, or other allergens can cause a sore throat.
        - Postnasal drip: Drip from the nose or sinuses can cause a sore throat.

        If you're experiencing a sore throat, it's important to see a doctor or pharmacist, especially if it's severe or accompanied by other symptoms like fever, chills, or difficulty swallowing. They can help diagnose the underlying cause and re

In [ ]:
#to take real time input and test model
print("Health Assistant Chatbot")
print("Type your health question below. Type 'quit' to exit.")

user_query = ""

while user_query != "q":

    user_query = input("\n Hey Wassup please let us know how i can assist you: ")

    if user_query == "q":
        print("Thanks for choosing General Health Query Chatbot")
        break

    response = health_chatbot(user_query)
    print("\n \n Bot:", response)


Health Assistant Chatbot
Type your health question below. Type 'quit' to exit.

 Hey Wassup please let us know how i can assist you:  What causes a sore throat?


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 
 Bot: A sore throat can be caused by several factors, including viral infections like the common cold, flu, or strep throat, as well as bacterial infections like tonsillitis. Other potential causes include allergies, irritation from smoking or air pollution, and certain medications. If you're experiencing a sore throat, it's important to see a healthcare professional to determine the underlying cause and receive appropriate treatment.

 Hey Wassup please let us know how i can assist you: fffff


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 
 Bot: Hello! How can I assist you today?

 Hey Wassup please let us know how i can assist you: Is paracetamol safe for children?


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 
 Bot: Paracetamol may be used in children as directed by a doctor or pharmacist. It can help to reduce fever and relieve pain. However, it is important to follow the instructions on the label or as directed by your doctor. If you have any questions or concerns, it is always best to consult with a healthcare professional.
